In [3]:
import fitz
import io
from PIL import Image
import pytesseract
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
import time
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np

In [5]:
pdf_path = r'D:\КНИГИ\book_torr\pdf_text\1016 Bahaa E. A. Saleh, Malvin Carl Teich Fundamentals of Photonics 3rd ed. 2019.pdf'

In [1]:
def ocr_page(page_num: int) -> dict:
    doc = fitz.open(pdf_path)
    page = doc[page_num]

    pix = page.get_pixmap(dpi=300, alpha=False)
    img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("L")

    text = pytesseract.image_to_string(
        img,
        lang="rus+eng",
        config="--oem 3 --psm 6"
    )

    doc.close()
    return {
        "page_num": page_num + 1,
        "text": text
    }

In [ ]:
start = time.time()
doc = fitz.open(pdf_path)
page_count = doc.page_count
doc.close()

pages = []

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(ocr_page, i) for i in range(page_count)]
    
    for future in tqdm(as_completed(futures), total=len(futures)):
        pages.append(future.result())

pages.sort(key=lambda x: x["page_num"])
print(round(time.time() - start) / 60, 'мин')

In [23]:
def looks_like_math(text: str) -> bool:
    if not text:
        return False

    text = text.strip()

    if len(text) > 120:
        return False

    math_tokens = [
        "=", "∫", "π", "δ", "√", "∞",
        "sin", "cos", "tan", "exp",
        "dx", "dy", "dz",
        "∑", "±", "≤", "≥", "ω", "λ", "μ"
    ]

    if any(tok in text for tok in math_tokens):
        return True

    if re.search(r"\(\d+(\.\d+)*\)$", text):
        return True

    special_count = sum(ch in "=+-*/^_[]{}<>∫πδ√∞∑±≤≥ωλμ" for ch in text)
    if special_count >= 3:
        return True

    return False